# Megadetector v1000 Sagemaker Serverless Deployment

This notebook deploys megadetector v1000 to a Sagemaker serverless endpoint.  It is intended to be run in a SageMaker Notebook instance on the conda_pytorch_p10 kernel.

## Setup

In [3]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import time
import json
import base64
from datetime import datetime

## Initialize AWS Session

In [2]:
sess = boto3.Session()
sm = sess.client('sagemaker')
region = sess.region_name
account = boto3.client('sts').get_caller_identity().get('Account')

## Get IAM Role

Note: Ensure the IAM role has:

- `AmazonS3FullAccess`
- `AmazonSageMakerFullAccess`


In [4]:
role = sagemaker.get_execution_role()
print(f"Using role: {role}")

Using role: arn:aws:iam::830244800171:role/service-role/AmazonSageMaker-ExecutionRole-20210125T212674


## Create ECR repository

In [5]:
# Create ECR repository if it doesn't exist
registry_name = "mdv1000-sagemaker-serverless"
ecr = boto3.client('ecr')

try:
    ecr.create_repository(repositoryName=registry_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    pass

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
sha256:a82a58f1001b7e735bad407a50b489879aa1776bc335fb68ebfd5e54d3e9c30f
The push refers to repository [830244800171.dkr.ecr.us-west-2.amazonaws.com/mdv1000-sagemaker-serverless]

bf18a086: Preparing 
9388415e: Preparing 
67663692: Preparing 
747858b9: Preparing 
8fd9973a: Preparing 
274be4f3: Preparing 
d0aa80cd: Preparing 
18dd62d2: Preparing 
c8d16465: Preparing 
latest: digest: sha256:05eed9057d36e3b770f4e34dec2a17af5d64e50dd1f54c90916557ec81daf6d2 size: 2419
Container pushed to: 830244800171.dkr.ecr.us-west-2.amazonaws.com/mdv1000-sagemaker-serverless:latest


## Build and upload container

Builds the the inference container with megadetector v1000 and the sagemaker handler and uploads it to ECR.  This step takes several minutes after it prints the 'Login Succeeded' message.  Be patient and trust the process.

In [8]:
# flag to avoid timely image builds
should_create = False

if should_create:
    # Get auth token and login to ECR
    !aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account}.dkr.ecr.{region}.amazonaws.com
    
    # Build container
    !docker build -q -t {registry_name} -f Dockerfile .
    
    # Tag and push to ECR
    image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{registry_name}:latest"
    !docker tag {registry_name} {image_uri}
    !docker push {image_uri}
    
    print(f"Container pushed to: {image_uri}")

## Create Sagemaker model

In [11]:
model_prefix = "mdv1000-0-0-redwood"

# Check if model already exists
model_already_created = False
for model_def in sm.list_models()['Models']:
    if model_prefix == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True

# Create model if it doesn't exist
if not model_already_created:
    create_model_response = sm.create_model(
        ModelName=model_prefix,
        ExecutionRoleArn=role,
        PrimaryContainer={
            "Image": image_uri,
            "Environment": {
                "SAGEMAKER_PROGRAM": "serve.py"
            }
        }
    )

print(f"Model ARN: {create_model_response['ModelArn']}")

Model ARN: arn:aws:sagemaker:us-west-2:830244800171:model/mdv1000-0-0-redwood


## Create Sagemaker realtime endpoint config

In [14]:
# Create realtime and batch endpoint configuration
realtime_endpoint_config_name = f"{model_prefix}-realtime-config"

realtime_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=realtime_endpoint_config_name,
    ProductionVariants=[
        {
            "ModelName": model_prefix,
            "VariantName": "AllTraffic",
            "ServerlessConfig": {
                "MemorySizeInMB": 6144,  # 6GB memory
                "MaxConcurrency": 20       # Maximum concurrent invocations
            }
        }
    ]
)
print(f"Realtime endpoint config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")

Realtime endpoint config ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint-config/mdv1000-0-0-redwood-realtime-config


## Create realtime endpoint

In [ ]:
# Create realtime endpoint
realtime_endpoint_name = f"{model_prefix}-concurrency-20"
create_realtime_endpoint_response = sm.create_endpoint(
    EndpointName=realtime_endpoint_name,
    EndpointConfigName=realtime_endpoint_config_name
)

print(f"Endpoint ARN: {create_realtime_endpoint_response['EndpointArn']}")

# Wait for endpoint creation
resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
realtime_status = resp['EndpointStatus']
print(f"Status: {realtime_status}")

while realtime_status == 'Creating':
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
    realtime_status = resp['EndpointStatus']
    print(f"Status: {realtime_status}")
    if realtime_status == 'Failed':
        realtime_failure_reason = resp.get('FailureReason', 'No failure reason provided')
        print(f"Realtime endpoint deployment failed: {realtime_failure_reason}")
        break

# Get CloudWatch logs for the endpoint
logs = boto3.client('logs')

print(f"Realtime Arn: {resp['EndpointArn']}")
print(f"Realtime endpoint final status: {realtime_status}")
if realtime_status == 'Failed':
    realtime_log_group = f"/aws/sagemaker/Endpoints/{realtime_endpoint_name}"
    try:
        log_streams = logs.describe_log_streams(logGroupName=realtime_log_group)
        for stream in log_streams['logStreams']:
            print(f"\nLog stream: {stream['logStreamName']}")
            realtime_events = logs.get_log_events(logGroupName=realtime_log_group, logStreamName=stream['logStreamName'])
            for event in realtime_events['events']:
                print(event['message'])
    except Exception as e:
        print(f"Error fetching logs: {str(e)}")

## Create batch endpoint config

Flip the `create_batch_endpoint_config` flag if not needed.

In [ ]:
batch_endpoint_config_name = f"{model_prefix}-batch-config"

# Disable batch endpoint config creation if not needed
create_batch_endpoint_config = False

if create_batch_endpoint_config:
    batch_endpoint_config_response = sm.create_endpoint_config(
        EndpointConfigName=batch_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_prefix,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 6144,  # 6GB memory
                    "MaxConcurrency": 80       # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Batch endpoint config ARN: {batch_endpoint_config_response['EndpointConfigArn']}")

## Create batch endpoint

Flip the `create_batch_endpoint` flag if not needed.

In [ ]:
# Disable batch endpoint config creation if not needed
create_batch_endpoint = False

# Create batch endpoint if needed
batch_endpoint_name = f"{model_prefix}-concurrency-80"
if create_batch_endpoint:
    create_batch_endpoint_response = sm.create_endpoint(
        EndpointName=batch_endpoint_name,
        EndpointConfigName=batch_endpoint_config_name
    )

    print(f"Batch Endpoint ARN: {create_batch_endpoint_response['EndpointArn']}")

    # Wait for batch endpoint creation
    batch_resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
    batch_status = batch_resp['EndpointStatus']
    print(f"Status: {batch_status}")

    while batch_status == 'Creating':
        time.sleep(60)
        batch_resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
        batch_status = resp['EndpointStatus']
        print(f"Status: {batch_status}")
        if batch_status == 'Failed':
            failure_reason = resp.get('FailureReason', 'No failure reason provided')
            print(f"Batch endpoint deployment failed: {failure_reason}")
            break
    print(f"Batch Arn: {resp['EndpointArn']}")
    print(f"Batch endpoint final status: {batch_status}")
    if batch_status == 'Failed':
        batch_log_group = f"/aws/sagemaker/Endpoints/{batch_endpoint_config_name}"
        try:
            log_streams = logs.describe_log_streams(logGroupName=batch_log_group)
            for stream in log_streams['logStreams']:
                print(f"\nLog stream: {stream['logStreamName']}")
                batch_events = logs.get_log_events(logGroupName=batch_log_group, logStreamName=stream['logStreamName'])
                for event in batch_events['events']:
                    print(event['message'])
        except Exception as e:
            print(f"Error fetching logs: {str(e)}")